# Pipeline de hitos 3D (.glb) · Muro de Boxes (Google Colab · GPU L4)

Genera hitos icónicos de los 24 circuitos en **Google Colab con GPU L4 (24 GB VRAM)** usando **Tencent Hunyuan3D-2 completo** (malla de alta precisión + **texturas PBR reales multivista**) y los optimiza al contrato PWA del juego sin perder UVs:

- **Geometría:** decimado limpio **antes de texturizar** (`FaceReducer` a ~5 000 caras) para que la textura UV se proyecte nítida sobre la malla low-poly final.
- **Texturas:** WebP de hasta **512 × 512 px** (baja automáticamente a `256 px` solo si el archivo supera 300 KB).
- **Compresión:** **Draco**, objetivo **< 300 KB** por hito.
- **Normalización:** base apoyada en `y = 0` y centrada en `X/Z` con `gltf-transform center --pivot below` (el juego escala automáticamente la altura a `spec.height` en `src/ui/map3d.ts`).

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU L4***.

In [ ]:
# 1. Comprobar GPU L4 e instalar Tencent Hunyuan3D-2 (ShapeGen + PaintPipeline) y gltf-transform
import os, glob, shutil, subprocess, sys, torch

assert torch.cuda.is_available(), "No hay GPU activa. Ve a Entorno de ejecución -> Cambiar tipo -> GPU L4."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU detectada: {gpu_name} ({vram_gb:.1f} GB VRAM)")
if vram_gb < 15:
    print("AVISO: Se recomienda GPU L4 (24 GB) para ejecutar ShapeGen + PaintPipeline con holgura.")

# Clonar Hunyuan3D-2 si aún no está en /content
REPO = "/content/Hunyuan3D-2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", REPO], check=True)

# Dependencias Python (aprovechando PyTorch+CUDA preinstalados de Colab; transformers 4.49.0 evita el cambio de claves DINOv2 de v5)
!pip -q install ninja pybind11 pymeshlab xatlas pygltflib trimesh "pillow>=12.3,<13" rembg onnxruntime-gpu einops omegaconf "transformers==4.49.0" "diffusers==0.32.2" accelerate safetensors huggingface_hub
!pip -q install -r /content/Hunyuan3D-2/requirements.txt "pillow>=12.3,<13" "transformers==4.49.0" "diffusers==0.32.2"
!pip -q install -e /content/Hunyuan3D-2 --no-build-isolation --no-deps

# Compilar extensiones CUDA/C++ necesarias para pintar texturas PBR reales (Hunyuan3DPaintPipeline)
print("Compilando custom_rasterizer (tarda ~1-2 min solo la primera vez)...")
!cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && pip -q install -e . --no-build-isolation

print("Compilando differentiable_renderer...")
!cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && python setup.py -q build_ext --inplace
# En Python 3.12 algunos entornos generan mesh_processor.cpython-*.so; creamos copia compatible si hace falta
dr_dir = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
for so in glob.glob(os.path.join(dr_dir, "mesh_processor*.so")):
    plain = os.path.join(dr_dir, "mesh_processor.so")
    if so != plain and not os.path.exists(plain):
        shutil.copy2(so, plain)

# Herramienta CLI para WebP + Draco + pivote sin perder UVs
!npm install -g --silent @gltf-transform/cli

# Verificar imports en un subproceso limpio (evita choque con la versión antigua de PIL cargada al arrancar Colab)
subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0, '/content/Hunyuan3D-2'); "
     "from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover, DegenerateFaceRemover, FaceReducer; "
     "from hy3dgen.texgen import Hunyuan3DPaintPipeline; "
     "print('Entorno L4 listo: Hunyuan3D-2 (ShapeGen + PaintPipeline) y gltf-transform instalados correctamente.')"],
    check=True
)

In [ ]:
# 2. Elegir modo de trabajo y preparar imágenes (o modelos .glb externos)
#
# MODO:
#   - "imagen_a_3d"    : Sube tus fotos nombradas <circuito>__<hito>.jpg/.png (p. ej. suzuka__noria.jpg,
#                        spielberg__toro_acero.png, austin__torre_cota.jpg) y Hunyuan3D-2 genera los .glb.
#   - "demo_redbull"   : Prueba rápida con 1 clic: descarga fotos limpias de Wikimedia para
#                        spielberg__toro_acero y austin__torre_cota para validar la calidad en tu L4.
#   - "solo_optimizar" : Sube .glb/.obj creados fuera (Blender, Meshy, Tripo3D...) nombrados <circuito>__<hito>.glb.

MODO = "imagen_a_3d"   # <-- Cambia a "demo_redbull" para probar ya, o "solo_optimizar" para .glb externos

# Parámetros de calidad y presupuesto PWA
TARGET_TRIS = 5000     # Caras tras FaceReducer ANTES de pintar texturas (óptimo para WebGL móvil: 4000-6000)
STEPS = 35             # Pasos de difusión de forma (30-50; en L4 35 pasos tarda ~15-20 s por hito)
OCTREE_RES = 380       # Resolución de malla inicial antes de reducir (380 da aristas y huecos muy limpios)
MAX_TEX_PX = 512       # Textura WebP máxima (baja a 256 solo si el .glb final supera MAX_KB)
MAX_KB = 300           # Límite de tamaño por hito en KB
SEED = 42

import os, urllib.request
from PIL import Image
from IPython.display import display

for d in ("/content/input_images", "/content/work_cutouts", "/content/work_shapes", "/content/raw_models", "/content/out_landmarks"):
    os.makedirs(d, exist_ok=True)

# Limpiar entradas previas de la sesión para no mezclar tandas
for folder in ("/content/input_images", "/content/work_cutouts", "/content/work_shapes", "/content/raw_models"):
    for fn in os.listdir(folder):
        os.remove(os.path.join(folder, fn))

if MODO == "demo_redbull":
    # Fotos de referencia con recorte limpio (izquierda, arriba, derecha, abajo en 0..1)
    DEMO_URLS = [
        ("spielberg__toro_acero.jpg",
         "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2f/Red_Bull_Ring_-_Bull_sculpture.jpg/1280px-Red_Bull_Ring_-_Bull_sculpture.jpg",
         (0.08, 0.08, 0.92, 0.94)),
        ("austin__torre_cota.jpg",
         "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3c/Circuit_of_the_Americas_Observation_Tower_2012.jpg/960px-Circuit_of_the_Americas_Observation_Tower_2012.jpg",
         (0.18, 0.02, 0.82, 0.98)),
    ]
    req_headers = {"User-Agent": "MuroDeBoxesF1/1.0 (Colab L4 landmark pipeline)"}
    for fn, url, crop in DEMO_URLS:
        dst = os.path.join("/content/input_images", fn)
        req = urllib.request.Request(url, headers=req_headers)
        with urllib.request.urlopen(req, timeout=30) as r:
            open(dst, "wb").write(r.read())
        im = Image.open(dst).convert("RGB")
        w, h = im.size
        im = im.crop((int(w*crop[0]), int(h*crop[1]), int(w*crop[2]), int(h*crop[3])))
        im.save(dst, quality=95)
        print(f"Descargada y recortada: {fn} ({im.size[0]}x{im.size[1]})")
        display(im.resize((260, int(260 * im.size[1] / im.size[0]))))

elif MODO == "imagen_a_3d":
    from google.colab import files
    print("Sube una o varias fotos nombradas <circuito>__<hito>.jpg o .png (con doble guion bajo).")
    print("Ejemplos: suzuka__noria.jpg | spielberg__toro_acero.png | austin__torre_cota.jpg | lasvegas__esfera.jpg")
    print("Consejo: cuanto más centrado, nítido y despejado esté el monumento (3/4 frontal), mejor saldrá el 3D.")
    uploaded = files.upload()
    for fn, data in uploaded.items():
        dst = os.path.join("/content/input_images", fn)
        open(dst, "wb").write(data)
        im = Image.open(dst)
        print(f"Cargada: {fn} ({im.size[0]}x{im.size[1]})")
        display(im.resize((240, max(1, int(240 * im.size[1] / im.size[0])))))

elif MODO == "solo_optimizar":
    from google.colab import files
    print("Sube tus archivos .glb/.obj nombrados <circuito>__<hito>.glb (p. ej. suzuka__noria.glb):")
    uploaded = files.upload()
    for fn, data in uploaded.items():
        open(os.path.join("/content/raw_models", fn), "wb").write(data)
        print(f"Subido: {fn} ({len(data)/1024:.1f} KB)")
else:
    raise ValueError(f"MODO no válido: {MODO}")

In [ ]:
# 3. Generar mallas 3D + texturas PBR con Tencent Hunyuan3D-2 (auto-adaptado a T4 12GB RAM y L4 53GB RAM)
#    Cada fase (rembg -> ShapeGen -> PaintPipeline) corre en su propio proceso limpio para liberar el 100% de RAM y VRAM.

import gc, glob, json, os, subprocess, sys, time
import numpy as np
from PIL import Image
from IPython.display import display

if "/content/Hunyuan3D-2" not in sys.path:
    sys.path.insert(0, "/content/Hunyuan3D-2")

os.makedirs("/content/work_cutouts", exist_ok=True)
os.makedirs("/content/work_shapes", exist_ok=True)
os.makedirs("/content/raw_models", exist_ok=True)

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3)
USE_MINI = ram_gb < 24.0
print(f"RAM de sistema detectada: {ram_gb:.1f} GB -> ShapeGen: {'Hunyuan3D-2mini (bajo consumo RAM)' if USE_MINI else 'Hunyuan3D-2 completo (L4/A100)'}")

if MODO == "solo_optimizar":
    print("MODO='solo_optimizar': saltando generación IA. Pasa a la celda 4.")
else:
    img_paths = sorted(
        p for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp")
        for p in glob.glob(os.path.join("/content/input_images", ext))
    )
    assert img_paths, "No hay imágenes en /content/input_images. Ejecuta antes la celda 2."

    # ---------- FASE 3.0: RECORTAR FONDO EN SUBPROCESO (libera 100% de RAM al terminar) ----------
    cutout_script = """
import glob, os, sys
import numpy as np
from PIL import Image
sys.path.insert(0, '/content/Hunyuan3D-2')
try:
    from hy3dgen.rembg import BackgroundRemover
    rembg_fn = BackgroundRemover()
except Exception:
    from rembg import remove as rembg_fn
for p in sorted(p for ext in ('*.png','*.jpg','*.jpeg','*.webp') for p in glob.glob(os.path.join('/content/input_images', ext))):
    stem = os.path.splitext(os.path.basename(p))[0]
    if '__' not in stem:
        continue
    im = Image.open(p).convert('RGBA')
    if np.array(im.getchannel('A')).min() > 240:
        im = rembg_fn(im.convert('RGB')).convert('RGBA')
    bbox = im.getchannel('A').getbbox()
    if bbox:
        im = im.crop(bbox)
    side = int(max(im.size) * 1.18)
    canvas = Image.new('RGBA', (side, side), (0, 0, 0, 0))
    canvas.paste(im, ((side - im.size[0]) // 2, (side - im.size[1]) // 2), im)
    if side > 1024:
        canvas = canvas.resize((1024, 1024), Image.LANCZOS)
    canvas.save(os.path.join('/content/work_cutouts', f'{stem}.png'))
    print('Recorte OK:', stem)
"""
    subprocess.run([sys.executable, "-c", cutout_script], check=True)

    cutout_paths = sorted(glob.glob("/content/work_cutouts/*.png"))
    assert cutout_paths, "Ninguna imagen tenía el formato <circuito>__<hito>.jpg/.png."
    print("--- Vista previa de recortes ---")
    for cp in cutout_paths:
        stem = os.path.splitext(os.path.basename(cp))[0]
        cut = Image.open(cp).convert("RGBA")
        prev = Image.new("RGBA", (200, 200), (28, 34, 48, 255))
        prev.alpha_composite(cut.resize((200, 200), Image.LANCZOS))
        print(stem)
        display(prev.convert("RGB"))

    # ---------- FASE 3A: GEOMETRÍA ----------
    import torch, trimesh
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover, DegenerateFaceRemover, FaceReducer

    def reducir_caras(mesh, target):
        try:
            mesh = FloaterRemover()(mesh)
            mesh = DegenerateFaceRemover()(mesh)
            return FaceReducer()(mesh, max_facenum=target)
        except Exception as e:
            print(f"  (FaceReducer no disponible en este Python: {e}; usando decimado trimesh/fast_simplification)")
            return mesh.simplify_quadric_decimation(face_count=target)

    if USE_MINI:
        print("[Fase 3A] Cargando Hunyuan3D-2mini (fp16, ligero para RAM 12GB)...")
        pipe_shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            "tencent/Hunyuan3D-2mini", subfolder="hunyuan3d-dit-v2-mini", variant="fp16"
        )
    else:
        print("[Fase 3A] Cargando Hunyuan3D-2 completo en GPU L4...")
        pipe_shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")

    for cp in cutout_paths:
        stem = os.path.splitext(os.path.basename(cp))[0]
        t0 = time.time()
        print(f"Generando geometría de {stem} ({STEPS} pasos, octree={OCTREE_RES})...")
        im = Image.open(cp).convert("RGBA")
        mesh = pipe_shape(
            image=im, num_inference_steps=STEPS, octree_resolution=OCTREE_RES, generator=torch.manual_seed(SEED)
        )[0]
        raw_f = len(mesh.faces)
        mesh = reducir_caras(mesh, TARGET_TRIS)
        out_p = os.path.join("/content/work_shapes", f"{stem}.glb")
        mesh.export(out_p)
        print(f"  -> {stem}: {raw_f} -> {len(mesh.faces)} tris en {time.time()-t0:.1f} s")

    del pipe_shape
    gc.collect()
    torch.cuda.empty_cache()

    # ---------- FASE 3B: TEXTURAS PBR MULTIVISTA ----------
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    print("\n[Fase 3B] Cargando Hunyuan3D-2 PaintPipeline en GPU L4...")
    pipe_paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")

    for sp in sorted(glob.glob("/content/work_shapes/*.glb")):
        stem = os.path.splitext(os.path.basename(sp))[0]
        cp = os.path.join("/content/work_cutouts", f"{stem}.png")
        out_raw = os.path.join("/content/raw_models", f"{stem}.glb")
        mesh = trimesh.load(sp, force="mesh")
        t0 = time.time()
        print(f"Pintando texturas PBR de {stem}...")
        try:
            painted = pipe_paint(mesh, image=Image.open(cp).convert("RGBA"))
            painted.export(out_raw)
            print(f"  -> {stem} texturizado en {time.time()-t0:.1f} s ({os.path.getsize(out_raw)/1024:.0f} KB bruto)")
        except Exception as e:
            print(f"  AVISO: falló el pintado de {stem} ({e}); guardando malla base sin textura.")
            mesh.export(out_raw)
        torch.cuda.empty_cache()

    del pipe_paint
    gc.collect()
    torch.cuda.empty_cache()
    print("\nGeneración completada. Ejecuta la celda 4 para comprimir con WebP + Draco (<300 KB).")

In [ ]:
# 4. Optimizar al contrato PWA SIN perder UVs ni texturas (pivote y=0 + WebP 512px + Draco <300 KB)
import glob, json, os, shutil, subprocess, tempfile
import trimesh

TARGET_TRIS = globals().get("TARGET_TRIS", 5000)
MAX_TEX_PX = globals().get("MAX_TEX_PX", 512)
MAX_KB = globals().get("MAX_KB", 300)

# Limpiar salida anterior
if os.path.isdir("/content/out_landmarks"):
    shutil.rmtree("/content/out_landmarks")
os.makedirs("/content/out_landmarks", exist_ok=True)

sources = sorted(glob.glob("/content/raw_models/*.glb") + glob.glob("/content/raw_models/*.obj"))
assert sources, "No hay modelos en /content/raw_models. Ejecuta antes las celdas 2 y 3."

manifest = {"version": 1, "glb": {}}
resumen = []

def contar_tris(path):
    try:
        sc = trimesh.load(path, force="scene")
        return sum(len(g.faces) for g in sc.geometry.values() if isinstance(g, trimesh.Trimesh))
    except Exception:
        return -1

for src in sources:
    base = os.path.splitext(os.path.basename(src))[0]
    if "__" not in base:
        print(f"Saltando {src}: usa el formato <circuito>__<hito>.glb")
        continue
    cid, hid = base.split("__", 1)
    out_dir = os.path.join("/content/out_landmarks", cid)
    os.makedirs(out_dir, exist_ok=True)
    out_glb = os.path.join(out_dir, f"{hid}.glb")

    with tempfile.TemporaryDirectory() as td:
        cur = src
        # Si es .obj externo, convertirlo a .glb conservando materiales/UVs
        if src.lower().endswith(".obj"):
            conv = os.path.join(td, "conv.glb")
            trimesh.load(src, force="scene").export(conv, file_type="glb")
            cur = conv

        n_before = contar_tris(cur)

        # 1) Centrar en X/Z y apoyar exactamente la base en y = 0 (conserva UVs y texturas 100%)
        centered = os.path.join(td, "centered.glb")
        r = subprocess.run(["gltf-transform", "center", cur, centered, "--pivot", "below"], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(centered):
            cur = centered

        # 2) Si un .glb externo viene muy denso (>1.4x TARGET_TRIS), decimar con meshoptimizer preservando UVs
        if n_before > int(TARGET_TRIS * 1.4):
            ratio = max(0.05, min(0.95, TARGET_TRIS / float(n_before)))
            simp = os.path.join(td, "simp.glb")
            r = subprocess.run(
                ["gltf-transform", "simplify", cur, simp, "--ratio", f"{ratio:.4f}", "--error", "0.01"],
                capture_output=True, text=True
            )
            if r.returncode == 0 and os.path.exists(simp):
                cur = simp

        n_after = contar_tris(cur)

        # 3) Redimensionar textura + WebP + compresión Draco (en gltf-transform v4 resize y webp van separados)
        for tex_px, qual in ((MAX_TEX_PX, 82), (384, 76), (256, 72)):
            res_glb = os.path.join(td, f"res_{tex_px}.glb")
            webp_glb = os.path.join(td, f"webp_{tex_px}.glb")
            r1 = subprocess.run(
                ["gltf-transform", "resize", cur, res_glb, "--width", str(tex_px), "--height", str(tex_px)],
                capture_output=True, text=True
            )
            step1 = res_glb if (r1.returncode == 0 and os.path.exists(res_glb)) else cur
            r2 = subprocess.run(
                ["gltf-transform", "webp", step1, webp_glb, "--quality", str(qual)],
                capture_output=True, text=True
            )
            step2 = webp_glb if (r2.returncode == 0 and os.path.exists(webp_glb)) else step1
            subprocess.run(
                ["gltf-transform", "draco", step2, out_glb, "--method", "edgebreaker"],
                check=True
            )
            kb = os.path.getsize(out_glb) / 1024.0
            used_px = tex_px
            if kb <= MAX_KB:
                break

    manifest["glb"][f"{cid}/{hid}"] = True
    estado = "OK" if kb <= MAX_KB else f"AVISO >{MAX_KB} KB"
    resumen.append((f"{cid}/{hid}", n_before, n_after, used_px, kb, estado))
    print(f"{cid}/{hid:20s} | {n_before:6d} -> {n_after:5d} tris | WebP {used_px}px | {kb:6.1f} KB | {estado}")

with open("/content/out_landmarks/manifest.json", "w", encoding="utf8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
    f.write("\n")

shutil.make_archive("/content/landmarks", "zip", "/content/out_landmarks")
print("\nPaquete listo: /content/landmarks.zip")

In [ ]:
# 5. Descargar landmarks.zip a tu PC
#    Descomprímelo en public/assets/landmarks/ del repo (si ya tenías hitos en manifest.json,
#    añade las nuevas claves "<circuito>/<hito>": true en lugar de borrar las anteriores).
from google.colab import files
files.download("/content/landmarks.zip")